# Vesuvius Surface Detection — Easy Inference Notebook

**Usage:** Point `INPUT_FOLDER` to any folder containing `.tif` 3D volumes and run all cells.

Outputs per volume:
- Segmentation mask (`.tif`)
- Axial slice visualization (`.png`)
- Surface probability heatmap (`.png`)

## Configuration

In [ ]:
# ==========================================
# CHANGE THESE PATHS
# ==========================================

INPUT_FOLDER  = '/raid/home/vikram_govt/Dikshant/gautam/cv/data/test_images'   # folder with .tif volumes
OUTPUT_FOLDER = '/raid/home/vikram_govt/Dikshant/gautam/cv/notebooks/output'   # results go here

# Optional: path to ground truth labels folder (set None to skip metrics)
GT_FOLDER = None  # e.g. '/raid/home/vikram_govt/Dikshant/gautam/cv/data/train_labels'

# Which models to use (set False to skip)
USE_CUSTOM_UNET = True
USE_NNUNET_ENSEMBLE = True

# Post-processing settings
T_LOW = 0.2       # hysteresis low threshold
T_HIGH = 0.83     # hysteresis high threshold
MIN_SIZE = 2000    # remove components smaller than this

## Setup

In [ ]:
import sys, os
PROJECT_DIR = '/raid/home/vikram_govt/Dikshant/gautam/cv'
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import numpy as np
import torch
import tifffile
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pathlib import Path
from glob import glob
import scipy.ndimage as ndi
from scipy.ndimage import binary_closing, binary_fill_holes
from skimage.morphology import remove_small_objects, ball

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

tif_files = sorted(glob(os.path.join(INPUT_FOLDER, '*.tif')))
print(f'Device: {device}')
print(f'Found {len(tif_files)} volumes in {INPUT_FOLDER}')
for f in tif_files:
    print(f'  {os.path.basename(f)}')

## Load Models

In [ ]:
from src.models.unet3d import UNet3D
from src.utils.utils import get_gaussian_3d

# ---- Custom 3D U-Net ----
model_unet = None
if USE_CUSTOM_UNET:
    ckpt_path = os.path.join(PROJECT_DIR, 'checkpoints', 'checkpoint_best.pth')
    if os.path.exists(ckpt_path):
        model_unet = UNet3D(1, 3, 32, [1,2,4,8,10], 2, True).to(device)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        model_unet.load_state_dict(ckpt['model_state_dict'])
        model_unet.eval()
        print(f'3D U-Net loaded (epoch {ckpt["epoch"]}, surface_dice={ckpt["best_surface_dice"]:.4f})')
    else:
        print(f'3D U-Net checkpoint not found at {ckpt_path}')

# ---- nnU-Net models ----
nnunet_predictors = {}
if USE_NNUNET_ENSEMBLE:
    os.environ['nnUNet_raw'] = os.path.join(PROJECT_DIR, 'nnUNet_data/nnUNet_raw')
    os.environ['nnUNet_preprocessed'] = os.path.join(PROJECT_DIR, 'nnUNet_data/nnUNet_preprocessed')
    os.environ['nnUNet_results'] = os.path.join(PROJECT_DIR, 'nnUNet_data/nnUNet_results')
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

    results_base = os.path.join(PROJECT_DIR, 'nnUNet_data/nnUNet_results/Dataset200_VesuviusSurface')
    model_configs = {
        'Default':  ('nnUNetTrainer__nnUNetPlans__3d_fullres', 0.40),
        'MPlans':   ('nnUNetTrainer_4000epochs__nnUNetResEncUNetMPlans__3d_fullres', 0.35),
        'LPlans':   ('nnUNetTrainer_4000epochs__nnUNetResEncUNetLPlans__3d_fullres', 0.25),
    }

    for name, (subdir, weight) in model_configs.items():
        model_dir = os.path.join(results_base, subdir)
        ckpt_file = os.path.join(model_dir, 'fold_all', 'checkpoint_best.pth')
        if os.path.exists(ckpt_file):
            pred = nnUNetPredictor(
                tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
                device=device, verbose=False,
            )
            pred.initialize_from_trained_model_folder(
                model_dir, use_folds=('all',), checkpoint_name='checkpoint_best.pth',
            )
            nnunet_predictors[name] = (pred, weight)
            print(f'nnU-Net {name} loaded (weight={weight})')
        else:
            print(f'nnU-Net {name} checkpoint not found, skipping')

print(f'\nModels ready: custom_unet={model_unet is not None}, nnunet_models={list(nnunet_predictors.keys())}')

## Inference + Post-Processing Functions

In [ ]:
# ---- 3D U-Net sliding window ----
def unet_inference(model, volume_np, patch_size=128, overlap=0.5):
    volume = torch.from_numpy(volume_np.astype(np.float32) / 255.0).unsqueeze(0).unsqueeze(0).to(device)
    step = int(patch_size * (1 - overlap))
    D, H, W = volume.shape[2:]
    gaussian = get_gaussian_3d(patch_size).to(device)
    output_sum = torch.zeros(3, D, H, W, device=device)
    weight_sum = torch.zeros(1, D, H, W, device=device)
    starts = lambda sz: sorted(set(list(range(0, max(sz-patch_size,0)+1, step)) + [max(sz-patch_size,0)]))
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        for d in starts(D):
            for h in starts(H):
                for w in starts(W):
                    patch = volume[:, :, d:d+patch_size, h:h+patch_size, w:w+patch_size]
                    probs = torch.softmax(model(patch)['logits'].float(), dim=1)[0]
                    output_sum[:, d:d+patch_size, h:h+patch_size, w:w+patch_size] += probs * gaussian
                    weight_sum[:, d:d+patch_size, h:h+patch_size, w:w+patch_size] += gaussian
    return (output_sum / weight_sum.clamp(min=1e-8)).cpu().numpy()

# ---- nnU-Net single model ----
def nnunet_inference(predictor, volume_np):
    img = volume_np.astype(np.float32)[np.newaxis]
    _, probs = predictor.predict_single_npy_array(
        img, {'spacing': [1.0, 1.0, 1.0]}, None, None, True
    )
    return probs  # (C, D, H, W)

# ---- Ensemble ----
def ensemble_nnunet(predictors_dict, volume_np):
    probs_sum = None
    for name, (predictor, weight) in predictors_dict.items():
        probs = nnunet_inference(predictor, volume_np)
        if probs_sum is None:
            probs_sum = weight * probs
        else:
            probs_sum += weight * probs
        torch.cuda.empty_cache()
    return probs_sum

# ---- Post-processing ----
def postprocess(surface_probs, t_low=0.2, t_high=0.83, min_size=2000):
    # Hysteresis thresholding
    strong = surface_probs >= t_high
    weak = surface_probs >= t_low
    if not strong.any():
        return np.zeros_like(surface_probs, dtype=np.uint8)
    struct = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct)
    # Closing
    mask = ndi.binary_closing(mask, structure=ndi.generate_binary_structure(3, 1), iterations=2)
    # Dust removal
    mask = remove_small_objects(mask, min_size=min_size)
    # Zero faces (3 voxels)
    for t in range(3):
        mask[t] = mask[-t-1] = False
        mask[:, t] = mask[:, -t-1] = False
        mask[:, :, t] = mask[:, :, -t-1] = False
    mask = remove_small_objects(mask, min_size=1000)
    # Close + fill
    mask = binary_closing(mask, structure=ball(3))
    for z in range(mask.shape[0]):
        if mask[z].any():
            mask[z] = binary_fill_holes(mask[z])
    mask = binary_fill_holes(mask)
    return mask.astype(np.uint8)

def to_3class(surface_mask, air_prob, papyrus_prob):
    result = np.zeros_like(surface_mask, dtype=np.uint8)
    result[surface_mask > 0] = 1
    non_surf = surface_mask == 0
    result[non_surf & (papyrus_prob > air_prob)] = 2
    return result

print('Functions ready.')

## Run Inference on All Volumes

In [ ]:
seg_cmap = ListedColormap(['black', 'red', 'dodgerblue'])

def visualize_volume(image, predictions, sample_id, output_dir, gt=None):
    """Save axial slice comparison + surface heatmap."""
    slices = [0.25, 0.5, 0.75]
    n_cols = 1 + (1 if gt is not None else 0) + len(predictions)
    fig, axes = plt.subplots(len(slices), n_cols, figsize=(4*n_cols, 4*len(slices)))
    if len(slices) == 1:
        axes = axes[np.newaxis, :]
    
    col = 0
    titles = ['CT Image']
    if gt is not None:
        titles.append('Ground Truth')
    titles += list(predictions.keys())
    
    for row, frac in enumerate(slices):
        idx = int(frac * image.shape[0])
        img_s = image[idx]
        
        col = 0
        axes[row, col].imshow(img_s, cmap='gray')
        axes[row, col].set_ylabel(f'Slice {idx}', fontsize=11)
        col += 1
        
        if gt is not None:
            axes[row, col].imshow(img_s, cmap='gray', alpha=0.5)
            axes[row, col].imshow(gt[idx], cmap=seg_cmap, alpha=0.5, vmin=0, vmax=2)
            col += 1
        
        for name, pred in predictions.items():
            axes[row, col].imshow(img_s, cmap='gray', alpha=0.5)
            axes[row, col].imshow(pred[idx], cmap=seg_cmap, alpha=0.5, vmin=0, vmax=2)
            col += 1
        
        for c in range(n_cols):
            axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
    
    for c, t in enumerate(titles):
        axes[0, c].set_title(t, fontsize=11, fontweight='bold')
    
    plt.suptitle(f'Volume: {sample_id}', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{sample_id}_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()


def visualize_surface_prob(image, prob_maps, sample_id, output_dir, gt=None):
    """Surface probability heatmap at mid-slice."""
    idx = image.shape[0] // 2
    n_cols = (1 if gt is not None else 0) + len(prob_maps)
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 5))
    if n_cols == 1:
        axes = [axes]
    
    col = 0
    if gt is not None:
        axes[col].imshow(image[idx], cmap='gray', alpha=0.5)
        axes[col].imshow(gt[idx] == 1, cmap='Reds', alpha=0.6)
        axes[col].set_title('GT Surface', fontweight='bold')
        axes[col].set_xticks([]); axes[col].set_yticks([])
        col += 1
    
    for name, prob in prob_maps.items():
        im = axes[col].imshow(prob[idx], cmap='hot', vmin=0, vmax=1)
        axes[col].set_title(name, fontweight='bold')
        axes[col].set_xticks([]); axes[col].set_yticks([])
        plt.colorbar(im, ax=axes[col], fraction=0.046)
        col += 1
    
    plt.suptitle(f'Surface Probability — {sample_id} (slice {idx})', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{sample_id}_surface_prob.png'), dpi=150, bbox_inches='tight')
    plt.show()

print('Visualization functions ready.')

In [ ]:
for tif_path in tif_files:
    sample_id = Path(tif_path).stem
    print(f'\n{"="*60}')
    print(f'  Processing: {sample_id}')
    print(f'{"="*60}')
    
    image = tifffile.imread(tif_path)
    print(f'  Volume: {image.shape} {image.dtype}')
    
    # Load GT if available
    gt = None
    if GT_FOLDER:
        gt_path = os.path.join(GT_FOLDER, f'{sample_id}.tif')
        if os.path.exists(gt_path):
            gt = tifffile.imread(gt_path)
            print(f'  Ground truth loaded')
    
    predictions = {}
    prob_maps = {}
    
    # ---- Custom 3D U-Net ----
    if model_unet is not None:
        print('  Running 3D U-Net...')
        probs = unet_inference(model_unet, image)
        surface_pp = postprocess(probs[1], T_LOW, T_HIGH, MIN_SIZE)
        pred_3c = to_3class(surface_pp, probs[0], probs[2])
        predictions['3D U-Net'] = pred_3c
        prob_maps['3D U-Net'] = probs[1]
        print(f'    Surface voxels: {(pred_3c == 1).sum():,}')
    
    # ---- nnU-Net Ensemble ----
    if nnunet_predictors:
        print('  Running nnU-Net ensemble...')
        probs_ens = ensemble_nnunet(nnunet_predictors, image)
        surface_pp = postprocess(probs_ens[1], T_LOW, T_HIGH, MIN_SIZE)
        pred_3c = to_3class(surface_pp, probs_ens[0], probs_ens[2])
        predictions['nnU-Net Ensemble'] = pred_3c
        prob_maps['nnU-Net Ensemble'] = probs_ens[1]
        print(f'    Surface voxels: {(pred_3c == 1).sum():,}')
    
    # ---- Metrics (if GT available) ----
    if gt is not None:
        from src.training.metrics import SegmentationMetrics
        print(f'\n  {"Model":<25} {"Surface Dice":>13} {"Mean Dice":>10}')
        print(f'  {"-"*50}')
        for name, pred in predictions.items():
            m = SegmentationMetrics(3, ['air','surface','papyrus'])
            m.update(torch.from_numpy(pred), torch.from_numpy(gt))
            r = m.compute()
            print(f'  {name:<25} {r["surface_dice"]:>13.4f} {r["mean_dice"]:>10.4f}')
    
    # ---- Save mask ----
    best_name = 'nnU-Net Ensemble' if 'nnU-Net Ensemble' in predictions else list(predictions.keys())[0]
    best_pred = predictions[best_name]
    out_path = os.path.join(OUTPUT_FOLDER, f'{sample_id}.tif')
    tifffile.imwrite(out_path, best_pred)
    print(f'\n  Saved mask: {out_path}')
    
    # ---- Visualize ----
    visualize_volume(image, predictions, sample_id, OUTPUT_FOLDER, gt=gt)
    visualize_surface_prob(image, prob_maps, sample_id, OUTPUT_FOLDER, gt=gt)
    
    torch.cuda.empty_cache()

print(f'\nDone! Results saved to {OUTPUT_FOLDER}')